In [1]:
import random
from datetime import datetime

import mysql.connector
from faker import Faker
from pydeequ.analyzers import *
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [2]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])

In [3]:
spark = SparkSession.builder.appName("PyDeequ Example").master("spark://localhost:7077").config("spark.sql.shuffle.partitions", "1").getOrCreate()

26/01/19 18:57:42 WARN Utils: Your hostname, MacBook-Pro-14.local resolves to a loopback address: 127.0.0.1; using 10.12.2.32 instead (on interface en0)
26/01/19 18:57:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 18:57:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/19 18:57:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/19 18:57:43 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/19 18:57:43 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [4]:
data = [{
    "id": i + 1,
    "name": fake.name(),
    "age": fake.random_int(min=20, max=65),
    "weigh": fake.random_int(min=10, max=250),
    "height": fake.random_int(min=10, max=250),
    "gender": random.choice(['남성', '여성']),
    "address": fake.address(),
    "job": fake.job(),
    "email": fake.email()
} for i in range(100)]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("weight", IntegerType(), True),
    StructField("height", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("address", StringType(), True),
    StructField("job", StringType(), True),
    StructField("email", StringType(), True)
])

In [5]:
df = spark.createDataFrame(data=data, schema=schema)

In [6]:
check = (
    Check(spark, CheckLevel.Error, "Basic data checks")
    .hasSize(lambda x: x == 100)
    .isComplete("id")
    .isComplete("name")
    .isComplete("age")
    .isComplete("gender")
    .isComplete("email")
)

result = (VerificationSuite(spark).onData(df).addCheck(check).run())

Python Callback server started!


ERROR:root:KeyboardInterrupt while sending command.                 (0 + 0) / 2]
Traceback (most recent call last):
  File "/opt/anaconda3/envs/enjoy-workreduce/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/enjoy-workreduce/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/anaconda3/envs/enjoy-workreduce/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
analysis_runner = AnalysisRunner(spark)
analysis_result = (
    analysis_runner.onData(df)
    .addAnalyzer(Size())
    .addAnalyzer(Completeness("id"))
    .addAnalyzer(Completeness("name"))
    .addAnalyzer(Completeness("age"))
    .addAnalyzer(Correlation("height", "weight"))
    .addAnalyzer(Completeness("gender"))
    .addAnalyzer(Completeness("address"))
    .addAnalyzer(Completeness("job"))
    .addAnalyzer(Completeness("email"))
    .run())

In [ ]:
result_df = AnalyzerContext.successMetricsAsDataFrame(spark, analysis_result) \
    .withColumn("run_name", lit("daily_batch")) \
    .withColumn("run_id", lit(f"daily_batch_{datetime.now().strftime('%Y%m%d%H%M%S')}")) \
    .withColumn("logical_datetime", lit(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"))
result_df.printSchema()

In [ ]:
connection = mysql.connector.connect(host="127.0.0.1", user="root", password="root", database="mmix")

In [ ]:
with connection.cursor() as cursor:
    insert_query = """
                   INSERT INTO etl_analysis_logs (run_name, run_id, logical_datetime, entity, instance, name, value)
                   VALUES (%(run_name)s, %(run_id)s, %(logical_datetime)s, %(entity)s, %(instance)s, %(name)s, %(value)s) ON DUPLICATE KEY
                   UPDATE value =
                   VALUES (value)
                   """
    cursor.executemany(insert_query, [row.asDict() for row in result_df.collect()])
    connection.commit()